In [ ]:
https://aact.ctti-clinicaltrials.org/downloads

In [2]:
required_aact_files = [
    "studies.txt",
    "sponsors.txt",
    "interventions.txt",
    "countries.txt",
    "designs.txt",
    "calculated_values.txt",
    "conditions.txt",
    "browse_conditions.txt",
    "brief_summaries.txt",
    "design_groups.txt",
    "outcomes.txt",
    "eligibilities.txt",
    "design_outcomes.txt",
    "outcome_analyses.txt"
]

# Data Quality Check Script: Explained

**Purpose:** This script acts as a "safety scanner" for our raw AACT text files before we load them into our analysis pipeline. It ensures we don't accidentally lose data due to formatting errors or messy content.

**What it does (Step-by-Step):**

1.  **Iterates through Files:** It automatically finds every `.txt` file in our `00_data` folder.

2.  **Structural Validation (The "Pipe Check"):**
    * It scans every line to count the pipe separators (`|`).
    * *Why?* If a line has too many pipes (e.g., a pipe accidentally typed in a description), it shifts all the data, ruining the row. This script catches those errors.

3.  **Correct Loading:**
    * It attempts to load the file using the "Golden Settings":
        * `sep='|'`: Pipe delimiter.
        * `quotechar='"'`: Respects quotes (fixing the "broken rows" issue we found earlier).
        * `dtype=str`: Reads everything as text first (safest method).

4.  **Content Analysis:**
    * **Whitespace:** Checks if columns have invisible spaces (e.g., `" Diabetes "` vs `"Diabetes"`).
    * **Mixed Types:** Checks if numeric columns contain text (e.g., `"1,200"` or `"18-35"`).

5.  **Reporting:**
    * It saves a full audit log to `data_quality_report_v2.txt` so we have a permanent record of our data health.

**Why use it?**
Running this ensures our cleaning script (`clean_aact_final.py`) is based on *facts*, not guesses. It proves our data integrity is solid before we start Machine Learning.

In [3]:
import pandas as pd
import csv
import sys
import os
import re
from datetime import datetime
from pathlib import Path

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [4]:
# --- PATH IDENTIFICATION LOGIC ---

# 1. Define the Project Root
# Start at the current directory (where this notebook is located)
current_dir = Path.cwd()
project_root = current_dir

# Search for 'src' to identify the root
while not (project_root / 'src').exists():
    if project_root == project_root.parent:
        # Hit the filesystem root without finding the project
        raise FileNotFoundError("Could not find project root containing 'src'")
    project_root = project_root.parent

# 2. Add Project Root to System Path
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# 3. Define Key Paths based on your requirements
# Raw data: /home/delaunan/code/delaunan/clintrialpredict/data/raw
RAW_DATA_PATH = project_root / "data" / "raw"

# Output (Cleaned files/Reports): /home/delaunan/code/delaunan/clintrialpredict/data
OUTPUT_PATH = project_root / "data"

# Report File Path
REPORT_FILE = OUTPUT_PATH / "data_quality_report_v2.txt"

# 4. Verification
print(f"Project Root: {project_root}")
print(f"Raw Data:     {RAW_DATA_PATH}")
print(f"Output Path:  {OUTPUT_PATH}")
print(f"Report File:  {REPORT_FILE}")

Project Root: /home/delaunan/code/delaunan/clintrialpredict
Raw Data:     /home/delaunan/code/delaunan/clintrialpredict/data/raw
Output Path:  /home/delaunan/code/delaunan/clintrialpredict/data
Report File:  /home/delaunan/code/delaunan/clintrialpredict/data/data_quality_report_v2.txt


In [5]:
# --- CONFIGURATION ---
DELIMITER = "|"
EXTENSION_TO_CHECK = ".txt"

# Buffer to hold report text for file writing
report_buffer = []

def log(message):
    """Helper function to print to console AND buffer for the report file."""
    print(message)
    report_buffer.append(message)

def save_report():
    """Writes the buffered log to the text file using the dynamic path."""
    try:
        # Ensure the output directory exists
        OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

        with open(REPORT_FILE, "w", encoding="utf-8") as f:
            f.write("\n".join(report_buffer))
        print(f"\n[SUCCESS] Full report saved to: {REPORT_FILE}")
    except Exception as e:
        print(f"\n[ERROR] Could not save report file: {e}")

In [6]:
# --- LOADING & CHECKING FUNCTIONS ---

def load_dataframe_safely(file_path, delimiter):
    """
    Loads a pipe-delimited file using the correct settings to handle quotes.
    Returns: DataFrame or None (if failed)
    """
    try:
        df = pd.read_csv(
            file_path,
            sep=delimiter,
            dtype=str,                  # 1. Read as text (prevent type errors)
            low_memory=False,           # 2. Read full file at once (prevent chunking errors)
            quotechar='"',              # 3. Handle quotes properly (fixes "broken rows")
            quoting=csv.QUOTE_MINIMAL,  # 4. Allow quotes to wrap fields containing pipes
            on_bad_lines='warn'         # 5. Warn if genuine corruption exists
        )
        return df
    except Exception as e:
        log(f"  [Critical Load Error] {e}")
        return None

def check_whitespace_issues(df):
    """Finds and lists columns that need trimming."""
    log("\n--- 1. HYGIENE: Whitespace Check ---")
    text_cols = df.select_dtypes(include=['object']).columns
    dirty_cols = []

    for col in text_cols:
        sample = df[col].dropna().head(5000).astype(str)
        if sample.empty: continue
        if (sample.str.len() != sample.str.strip().str.len()).any():
            dirty_cols.append(col)

    if dirty_cols:
        log(f"  [Action Needed] {len(dirty_cols)} columns have invisible spaces (e.g. ' Diabetes ').")
        log(f"  Columns: {dirty_cols[:5]}...")
    else:
        log("  [OK] No whitespace issues found.")

def check_mixed_types(df):
    """Checks if a column has both numbers and text (dangerous for analysis)."""
    log("\n--- 2. INTEGRITY: Mixed Data Types ---")

    for col in df.columns:
        # Skip if explicitly string
        if df[col].dtype == 'object':
            # Try to force numeric
            numeric_vals = pd.to_numeric(df[col], errors='coerce')
            num_count = numeric_vals.notna().sum()
            total_count = df[col].notna().sum()

            # If a column is mostly numbers (90%) but has some text, warn user
            if total_count > 0 and (num_count / total_count) > 0.9 and (num_count != total_count):
                log(f"  [Warning] Column '{col}' looks numeric but has {total_count - num_count} text values.")
                # Show non-numeric examples
                non_nums = df[col][numeric_vals.isna() & df[col].notna()]
                log(f"    Garbage examples: {non_nums.head(3).tolist()}")

def check_structure_and_counts(df, file_path):
    """Basic structure check using the DF shape."""
    log(f"\n--- 3. STRUCTURE: Row/Col Count ---")
    rows, cols = df.shape
    log(f"  Loaded {rows} rows and {cols} columns.")

    # Check for empty columns
    empty_cols = [col for col in df.columns if df[col].isnull().all()]
    if empty_cols:
        log(f"  [Info] {len(empty_cols)} columns are 100% EMPTY.")

def process_file(file_path_obj):
    """
    Orchestrates the checks for a single file.
    Args:
        file_path_obj (Path): The full pathlib object for the file
    """
    log("\n" + "="*60)
    log(f"PROCESSING: {file_path_obj.name}")
    log("="*60)

    if not file_path_obj.exists():
        log(f"Error: File not found at {file_path_obj}")
        return

    # Use the CORRECT loader
    df = load_dataframe_safely(file_path_obj, DELIMITER)

    if df is not None:
        check_structure_and_counts(df, file_path_obj)
        check_whitespace_issues(df)
        check_mixed_types(df)

In [7]:
# --- EXECUTION LOOP ---
# Reset buffer if you run this cell multiple times
report_buffer = []

# Define the specific list of files to audit
required_aact_files = [
    "studies.txt",
    "sponsors.txt",
    "interventions.txt",
    "countries.txt",
    "designs.txt",
    "calculated_values.txt",
    "conditions.txt",
    "browse_conditions.txt",
    "brief_summaries.txt",
    "design_groups.txt",
    "outcomes.txt",
    "eligibilities.txt",
    "design_outcomes.txt",
    "outcome_analyses.txt"
]

log(f"Starting FINAL Data Quality Scan at {datetime.now()}")

if not RAW_DATA_PATH.exists():
    log(f"CRITICAL ERROR: The folder '{RAW_DATA_PATH}' was not found.")
else:
    log(f"Targeting {len(required_aact_files)} specific AACT files in '{RAW_DATA_PATH}'...")

    found_count = 0
    missing_files = []

    # Iterate ONLY through the required list
    for filename in required_aact_files:
        # Construct the full path using pathlib
        file_path = RAW_DATA_PATH / filename

        if file_path.exists():
            process_file(file_path)
            found_count += 1
        else:
            # Log missing files so you know they were skipped
            missing_files.append(filename)

    # --- FINAL SUMMARY ---
    log("\n" + "="*60)
    log(f"EXECUTION SUMMARY")
    log("="*60)
    log(f"Processed: {found_count}/{len(required_aact_files)} files.")

    if missing_files:
        log(f"\n[WARNING] The following {len(missing_files)} required files were NOT found:")
        for missing in missing_files:
            log(f"  - {missing}")
    else:
        log("\n[SUCCESS] All required files were found and processed.")

print("\nProcessing complete. Run the next cell to save the report.")

Starting FINAL Data Quality Scan at 2026-01-08 09:28:08.261986
Targeting 14 specific AACT files in '/home/delaunan/code/delaunan/clintrialpredict/data/raw'...

PROCESSING: studies.txt

--- 3. STRUCTURE: Row/Col Count ---
  Loaded 564443 rows and 71 columns.
  [Info] 1 columns are 100% EMPTY.

--- 1. HYGIENE: Whitespace Check ---
  [Action Needed] 1 columns have invisible spaces (e.g. ' Diabetes ').
  Columns: ['ipd_access_criteria']...

--- 2. INTEGRITY: Mixed Data Types ---

PROCESSING: sponsors.txt

--- 3. STRUCTURE: Row/Col Count ---
  Loaded 902779 rows and 5 columns.

--- 1. HYGIENE: Whitespace Check ---
  [OK] No whitespace issues found.

--- 2. INTEGRITY: Mixed Data Types ---

PROCESSING: interventions.txt

--- 3. STRUCTURE: Row/Col Count ---
  Loaded 954821 rows and 5 columns.

--- 1. HYGIENE: Whitespace Check ---
  [Action Needed] 1 columns have invisible spaces (e.g. ' Diabetes ').
  Columns: ['name']...

--- 2. INTEGRITY: Mixed Data Types ---

PROCESSING: countries.txt

--- 

In [8]:
# Save the report to disk
save_report()


[SUCCESS] Full report saved to: /home/delaunan/code/delaunan/clintrialpredict/data/data_quality_report_v2.txt


# 🕵️ Deep Dive Diagnostic Script: Explained

**Purpose:** This script is a targeted diagnostic tool that investigates specific issues flagged by the initial `check_data_quality.py` report. It drills down into problematic files to reveal exactly *why* rows are broken or why data types are mixed, helping us design the perfect cleaning strategy.

**What it does (Step-by-Step):**

1.  **Structural Investigator ("Broken Row Finder"):**
    * **Target:** Files flagged with broken rows (e.g., `responsible_parties.txt`, `facilities.txt`).
    * **Action:** It reads the specific lines where column counts don't match the header.
    * **Output:** It extracts the raw text of these broken lines so we can see the root cause (e.g., a pipe `|` character hidden inside a job title like `"Doctor | Professor"`).
    * **Benefit:** Confirms if we need to change our loading parameters (e.g., enabling `quotechar='"'`).

2.  **Mixed Type Investigator ("Hidden String Finder"):**
    * **Target:** Numeric columns that contain text (e.g., `param_value` in `baseline_measurements.txt`).
    * **Action:** It tries to convert the column to numbers and isolates the values that fail.
    * **Output:** It lists the most frequent non-numeric patterns.
        * *Example Findings:* It distinguishes between fixable issues (like `"10,366"` or `"- 4.5"`) vs. complex data (like `"18-35"` ranges) vs. pure garbage (like `"units"`).
    * **Benefit:** Tells us exactly which cleaning functions to write (e.g., "strip commas", "remove spaces").

3.  **Unparseable Date Finder:**
    * **Target:** Date columns with errors (e.g., `anticipated_posting_date`).
    * **Action:** It finds values that fail standard datetime conversion or fall outside a reasonable year range (1900-2030).
    * **Output:** Shows typos like `"3333-12-01"` (placeholder) or `"1018-04-10"` (likely 2018).
    * **Benefit:** Allows us to build "Smart Fix" logic to correct typos instead of deleting them.

4.  **Reporting:**
    * It saves a summary log to `deep_dive_log.txt`.
    * It saves detailed CSV samples of the bad data to the `00_data_issues/` folder for manual inspection.

**Why use it?**
While the first script tells us *that* there is a problem, this script tells us *what* the problem is. It provides the evidence needed to write the final `clean_aact_final.py` script with confidence.

In [9]:
# --- CONFIGURATION ---
# We use the existing OUTPUT_PATH to create a subfolder for specific issue files
ISSUES_DIR = OUTPUT_PATH / "00_data_issues"
ISSUES_DIR.mkdir(parents=True, exist_ok=True)

LOG_FILE = OUTPUT_PATH / "deep_dive_log.txt"
DELIMITER = "|"

# Buffer for the log file
log_buffer = []

def log(message):
    """Prints to console and saves to buffer."""
    print(message)
    log_buffer.append(message)

def save_log():
    """Writes the full log to a file."""
    with open(LOG_FILE, "w", encoding="utf-8") as f:
        f.write("\n".join(log_buffer))
    print(f"\n[DONE] Diagnostic log saved to: {LOG_FILE}")

print(f"Diagnostics will read from: {RAW_DATA_PATH}")
print(f"Issue reports will save to: {ISSUES_DIR}")

Diagnostics will read from: /home/delaunan/code/delaunan/clintrialpredict/data/raw
Issue reports will save to: /home/delaunan/code/delaunan/clintrialpredict/data/00_data_issues


In [10]:
def investigate_broken_rows(filename):
    """
    Reads specific line numbers to see WHY they are broken.
    """
    file_path = RAW_DATA_PATH / filename
    log(f"\n--- INVESTIGATION: Broken Rows in '{filename}' ---")

    if not file_path.exists():
        log(f"  [Error] File not found: {file_path}")
        return

    try:
        with open(file_path, 'r', encoding='utf-8', errors='replace') as f:
            header = f.readline().strip()
            expected_cols = header.count(DELIMITER) + 1
            log(f"  Header ({expected_cols} cols): {header[:100]}...")

            bad_samples = []

            # Reset file pointer to start
            f.seek(0)

            for i, line in enumerate(f):
                cols = line.count(DELIMITER) + 1
                if cols != expected_cols:
                    # Save the bad line for analysis
                    clean_line = line.strip()
                    bad_samples.append({
                        "line_number": i + 1,
                        "actual_cols": cols,
                        "expected_cols": expected_cols,
                        "content": clean_line[:200] + "..." if len(clean_line) > 200 else clean_line
                    })
                    if len(bad_samples) >= 5: # Just get first 5 examples
                        break

            if bad_samples:
                log(f"  Found broken lines. Examples:")
                for b in bad_samples:
                    log(f"    Line {b['line_number']} (Cols {b['actual_cols']}): {b['content']}")

                # Save to CSV for user to inspect
                sample_df = pd.DataFrame(bad_samples)
                sample_path = ISSUES_DIR / f"broken_rows_{filename.replace('.txt', '.csv')}"
                sample_df.to_csv(sample_path, index=False)
                log(f"  [Saved] Full bad row details to: {sample_path}")
            else:
                log("  [Info] No broken rows found.")

    except Exception as e:
        log(f"  [Error] Failed to read file: {e}")

In [11]:
def investigate_mixed_types(filename, col_name):
    """
    Loads a specific column and finds values that are NOT numbers.
    """
    file_path = RAW_DATA_PATH / filename
    log(f"\n--- INVESTIGATION: Mixed Types in '{filename}' (Col: {col_name}) ---")

    if not file_path.exists():
        log(f"  [Error] File not found: {file_path}")
        return

    try:
        # Load only the specific column to save memory
        df = pd.read_csv(
            file_path,
            sep=DELIMITER,
            usecols=[col_name],
            dtype=str, # Read as string first
            quoting=csv.QUOTE_NONE,
            on_bad_lines='skip' # Skip broken rows for this check
        )

        # Try to convert to numeric
        numeric = pd.to_numeric(df[col_name], errors='coerce')

        # Find rows where conversion failed (Result is NaN) BUT original was NOT empty
        mask_bad = numeric.isna() & df[col_name].notna()
        bad_values = df[mask_bad]

        if not bad_values.empty:
            count = len(bad_values)
            log(f"  Found {count} non-numeric values in '{col_name}'.")

            # Get top frequent garbage (e.g., "10,000" vs "unknown")
            top_garbage = bad_values[col_name].value_counts().head(10)
            log(f"  Top non-numeric patterns:\n{top_garbage.to_string()}")

            # Save specific file
            out_name = f"mixed_types_{filename.replace('.txt', '')}_{col_name}.csv"
            bad_values.head(100).to_csv(ISSUES_DIR / out_name, index=False)
            log(f"  [Saved] Sample of bad values to: {ISSUES_DIR / out_name}")
        else:
            log("  [OK] Column appears cleanly numeric (or empty).")

    except Exception as e:
        log(f"  [Error] Could not check mixed types: {e}")

In [12]:
def investigate_bad_dates(filename, col_name):
    """
    Finds dates like '3333-12-01' or 'Pending'.
    """
    file_path = RAW_DATA_PATH / filename
    log(f"\n--- INVESTIGATION: Weird Dates in '{filename}' (Col: {col_name}) ---")

    if not file_path.exists():
        log(f"  [Error] File not found: {file_path}")
        return

    try:
        df = pd.read_csv(
            file_path,
            sep=DELIMITER,
            usecols=[col_name],
            dtype=str,
            quoting=csv.QUOTE_NONE,
            on_bad_lines='skip'
        )

        # Try to convert
        dates = pd.to_datetime(df[col_name], errors='coerce')

        # Find failures
        mask_bad = dates.isna() & df[col_name].notna()
        bad_values = df[mask_bad]

        if not bad_values.empty:
            log(f"  Found {len(bad_values)} unparseable dates.")
            log(f"  Examples: {bad_values[col_name].unique()[:10]}")
        else:
            # Check for logical outliers (e.g. year 3000)
            valid_dates = dates.dropna()
            future_mask = valid_dates > pd.Timestamp("2030-01-01")
            weird_future = valid_dates[future_mask]

            if not weird_future.empty:
                log(f"  Found {len(weird_future)} dates far in the future (Logical Errors).")
                log(f"  Examples: {weird_future.head().astype(str).tolist()}")
            else:
                log("  [OK] Dates look technically valid.")

    except Exception as e:
        log(f"  [Error] Checking dates: {e}")

In [13]:
# --- EXECUTION: TARGETING THE ISSUES ---
log_buffer = [] # Reset buffer
log("STARTING DEEP DIVE DIAGNOSTIC")

# 1. Structure Issues (Broken Rows)
# We check the files where your report found "Whitespace Issues"
# just to ensure the whitespace isn't caused by a column misalignment.
investigate_broken_rows("studies.txt")
investigate_broken_rows("interventions.txt")
investigate_broken_rows("design_groups.txt")

# 2. Mixed Data Types (Numbers vs Text)
# Your report specifically flagged 'outcome_analyses.txt' as having mixed types
# in the confidence interval columns.
investigate_mixed_types("outcome_analyses.txt", "ci_upper_limit_raw")
investigate_mixed_types("outcome_analyses.txt", "ci_lower_limit_raw")

# 3. Date Issues
# Checking critical dates in the main files from your required list.
# 'studies.txt' is the most important file, so we verify its dates are valid.
investigate_bad_dates("studies.txt", "start_date")
investigate_bad_dates("studies.txt", "completion_date")

# 'outcomes.txt' is in your list, checking for the issue found in previous audits
investigate_bad_dates("outcomes.txt", "anticipated_posting_date")

save_log()

STARTING DEEP DIVE DIAGNOSTIC

--- INVESTIGATION: Broken Rows in 'studies.txt' ---
  Header (71 cols): nct_id|nlm_download_date_description|study_first_submitted_date|results_first_submitted_date|disposi...
  Found broken lines. Examples:
    Line 1211 (Cols 72): NCT05560958||2022-09-27|||2025-09-16|2022-09-27|2022-09-30|ACTUAL|||||||2025-09-16|2025-09-17|ESTIMATED|2023-01-16|ACTUAL|2023-01-16|2025-09|2025-09-30|2030-07|ESTIMATED|2030-07-31|2030-07|ESTIMATED|2...
    Line 5711 (Cols 72): NCT06018818||2023-07-17|||2025-08-22|2023-08-29|2023-08-31|ACTUAL|||||||2025-08-22|2025-08-24|ACTUAL|2023-08-23|ACTUAL|2023-08-23|2025-07|2025-07-31|2025-07-21|ACTUAL|2025-07-21|2025-07-21|ACTUAL|2025...
    Line 5746 (Cols 73): NCT06840509||2025-02-18|||2025-08-22|2025-02-18|2025-02-21|ACTUAL|||||||2025-08-22|2025-08-24|ACTUAL|2025-07-23|ACTUAL|2025-07-23|2025-02|2025-02-28|2026-12|ESTIMATED|2026-12-31|2025-11|ESTIMATED|2025...
    Line 10284 (Cols 75): NCT03313960||2017-10-10|||2019-02-25|2017-10-13|

# Numeric Scale Check Script: Explained

**Purpose:** This script acts as a diagnostic tool to understand the true nature of "mixed" numeric columns before we clean them. It specifically helps us decide whether percentages like `57.5%` should be treated as `57.5` (0-100 scale) or `0.575` (0-1 scale).

**What it does (Step-by-Step):**

1.  **Loads Specific Columns:** It reads only the target column (e.g., `param_value`) from the specified file, using safe loading parameters to avoid crashes.

2.  **Isolates Pure Numbers:**
    * It uses regex to find rows that contain *only* digits and decimals (e.g., `12.5`, `100`).
    * It calculates statistics (Min, Max, Mean) on these pure numbers to establish a "baseline scale."
    * *Example:* If the max pure number is `10,000`, the scale is clearly not 0-1.

3.  **Isolates Percentages:**
    * It finds rows containing the `%` symbol.
    * It strips the symbol and converts the remaining text to numbers to see their range.
    * *Example:* `57.5%` -> `57.5`.

4.  **Logic & Verdict:**
    * It compares the "Pure Number" scale vs. the "Percentage" scale.
    * **Verdict Logic:**
        * If pure numbers are small (0-1) and percentages are large (0-100), it suggests we should **DIVIDE** by 100.
        * If pure numbers are large (>1) and percentages are large (0-100), it suggests we should just **STRIP** the symbol.

**Why use it?**
Blindly stripping `%` or blindly dividing by 100 can corrupt data. This script gives us the mathematical proof needed to choose the correct cleaning strategy for our final dataset.

In [14]:
# --- CONFIGURATION FOR THIS DIAGNOSTIC ---
# Defining params locally to ensure strict quoting for this specific check
LOAD_PARAMS = {
    "sep": "|",
    "dtype": str,
    "quotechar": '"',
    "quoting": csv.QUOTE_MINIMAL,
    "low_memory": False,
    "on_bad_lines": "warn"
}

def check_numeric_scale(filename, col_name):
    # Use the dynamic path defined in previous cells
    file_path = RAW_DATA_PATH / filename

    if not file_path.exists():
        print(f"[SKIP] {filename} not found in {RAW_DATA_PATH}")
        return

    print(f"\n--- Checking Scale: {filename} [{col_name}] ---")

    try:
        # Load only the specific column
        df = pd.read_csv(file_path, usecols=[col_name], **LOAD_PARAMS)

        # 1. Isolate Pure Numbers
        # (Rows that are just digits and dots, no text/symbols)
        pure_numbers = df[col_name][df[col_name].astype(str).str.match(r'^-?\d+\.?\d*$')]
        pure_floats = pd.to_numeric(pure_numbers)

        if pure_floats.empty:
            print("  No pure numbers found to compare against.")
        else:
            print(f"  Pure Number Stats (Sample size: {len(pure_floats)}):")
            print(f"    Min: {pure_floats.min()}")
            print(f"    Max: {pure_floats.max()}")
            print(f"    Mean: {pure_floats.mean():.2f}")

        # 2. Isolate Percentage Strings
        percent_strings = df[col_name][df[col_name].astype(str).str.contains(r'%')]

        if percent_strings.empty:
            print("  No '%' signs found in this column.")
        else:
            print(f"  Found {len(percent_strings)} rows with '%'.")
            print(f"    Examples: {percent_strings.head().tolist()}")

            # Logic Check
            clean_percents = pd.to_numeric(percent_strings.str.replace('%', ''), errors='coerce')
            print(f"    If we just strip '%', the range is: {clean_percents.min()} to {clean_percents.max()}")

            # Decision Helper
            if pure_floats.max() <= 1.0 and clean_percents.mean() > 1.0:
                print("\n  [VERDICT] Raw numbers are likely 0-1 (Decimals). You SHOULD divide % by 100.")
            elif pure_floats.mean() > 1.0:
                print("\n  [VERDICT] Raw numbers are likely 0-100 (Integers). You should STRIP % (Don't divide).")
            else:
                print("\n  [VERDICT] Scale is ambiguous. Manual review needed.")

    except Exception as e:
        print(f"Error: {e}")

# --- EXECUTION ---
# Check the problematic columns identified in logs
check_numeric_scale("outcome_measurements.txt", "dispersion_upper_limit_raw")
check_numeric_scale("outcome_measurements.txt", "param_value")
check_numeric_scale("baseline_measurements.txt", "param_value")


--- Checking Scale: outcome_measurements.txt [dispersion_upper_limit_raw] ---
  Pure Number Stats (Sample size: 766704):
    Min: -5860000.0
    Max: 785070000000000.0
    Mean: 2004745450.43
  Found 11 rows with '%'.
    Examples: ['0.7%', '0.7%', '57.5%', '48%', '100%']
    If we just strip '%', the range is: 0.7 to 100.0

  [VERDICT] Raw numbers are likely 0-100 (Integers). You should STRIP % (Don't divide).

--- Checking Scale: outcome_measurements.txt [param_value] ---
  Pure Number Stats (Sample size: 4618308):
    Min: -408000000.0
    Max: 875769282253977.0
    Mean: 418041699.04
  No '%' signs found in this column.

--- Checking Scale: baseline_measurements.txt [param_value] ---
  Pure Number Stats (Sample size: 2747482):
    Min: -663.0
    Max: 13920879614.0
    Mean: 13630.81
  No '%' signs found in this column.


# 🏭 Master Data Conversion Pipeline: Explained

**Purpose:** This is the final "production" script for data preparation. It processes every single file in our raw data folder (`00_data`) and converts them into a standardized, clean format ready for Machine Learning.

**What it does (Step-by-Step):**

1.  **Iterates through EVERYTHING:** It loops through all 50+ `.txt` files in the source folder. This ensures no part of the database is left behind in an old format.

2.  **Correct Loading (The "Safe Fix"):**
    * It applies the proven structural fix (`quotechar='"'`) to every file.
    * This guarantees that files with "pipes inside quotes" (like `facilities.txt`) are loaded correctly without breaking rows.

3.  **Surgical Cleaning (The "Smart Logic"):**
    * It checks a **Special Rules Registry**.
    * If a file is known to be "dirty" (e.g., `baseline_measurements.txt`), it applies specific cleaning functions:
        * **Numbers:** Removes commas, spaces, and `%` signs to create pure numbers.
        * **Dates:** Fixes typos (e.g., `1018` -> `2018`) and filters impossible years.
    * If a file is "standard," it skips deep cleaning to preserve data integrity.

4.  **Standard Hygiene:**
    * For *every* file, it trims invisible whitespace from text columns (e.g., `" Diabetes "` -> `"Diabetes"`).

5.  **Standard Export:**
    * It saves every file as a **Pipe-Delimited Text File (`.txt`)**.
    * **Crucial Feature:** It uses `quoting=csv.QUOTE_MINIMAL`. This ensures that if any field contains a pipe `|`, it is wrapped in quotes so it doesn't break the file structure again.

**Why use it?**
This script gives us a **single, unified folder (`00_data_ml_ready`)** where every file is structurally sound, clean, and formatted exactly the same way. This is the "Gold Standard" dataset for our project.

In [15]:
import pandas as pd
import csv
import numpy as np
from pathlib import Path

# --- 1. PATH SETUP ---
# Ensure we use the dynamic paths from previous cells
if 'RAW_DATA_PATH' not in locals():
    # Fallback if variable is lost
    RAW_DATA_PATH = Path.cwd().parent / "data" / "raw"
    OUTPUT_PATH = Path.cwd().parent / "data"

# Define Output Directory
CLEAN_DATA_PATH = OUTPUT_PATH / "cleaned"
CLEAN_DATA_PATH.mkdir(parents=True, exist_ok=True)

print(f"Input:  {RAW_DATA_PATH}")
print(f"Output: {CLEAN_DATA_PATH}")

# --- 2. STRUCTURAL SETTINGS ---
# These parameters are proven to fix the "Broken Rows" (pipes inside quotes)
AACT_LOAD_PARAMS = {
    "sep": "|",
    "dtype": str,                 # Read all as string first
    "quotechar": '"',             # Handles pipes inside quotes
    "quoting": csv.QUOTE_MINIMAL,
    "low_memory": False,
    "on_bad_lines": "skip"        # Skip the few truly malformed lines in studies.txt
}

Input:  /home/delaunan/code/delaunan/clintrialpredict/data/raw
Output: /home/delaunan/code/delaunan/clintrialpredict/data/cleaned


In [16]:
def clean_numeric_column(series):
    """
    Surgical Cleaning based on Audit Findings:
    - Removes spaces (Fixes '- 0.01')
    - Removes commas (Fixes '28,819')
    - Removes <, > (Fixes '<0.05')
    - Strips % but DOES NOT DIVIDE (Preserves 0-100 scale per diagnostic)
    """
    # Ensure string type
    clean = series.astype(str)

    # Remove all whitespace (handles the "minus space" typo)
    clean = clean.str.replace(r'\s+', '', regex=True)

    # Remove commas and inequality signs
    clean = clean.str.replace(r'[,<>]', '', regex=True)

    # Strip percentage signs (e.g., "57.5%" -> "57.5")
    clean = clean.str.replace('%', '', regex=False)

    # Convert to numeric, turning text garbage into NaN
    return pd.to_numeric(clean, errors='coerce')

def clean_date_column(series):
    """
    Smart Date Cleaning:
    - Fixes '10xx' typos (1018 -> 2018)
    - Forces datetime conversion
    - Filters 1900-2100 (Removes '3333' and 'Pending' text garbage)
    """
    # Fix specific typo found in provided_documents.txt
    series = series.astype(str).str.replace(r'^10(\d{2}-\d{2}-\d{2})', r'20\1', regex=True)

    # Coerce to datetime
    dates = pd.to_datetime(series, errors='coerce')

    # Filter logical range (drops year 3333)
    mask_valid = (dates.dt.year >= 1900) & (dates.dt.year <= 2100)
    return dates.where(mask_valid, pd.NaT)

In [17]:
# --- SPECIAL RULES REGISTRY ---
# Maps filenames to specific cleaning rules
SPECIAL_CONFIG = {
    "studies.txt": {
        "date_cols": ["study_first_submitted_date", "results_first_submitted_date", "completion_date", "start_date"]
    },
    "baseline_measurements.txt": {
        "numeric_cols": ["param_value", "dispersion_value"]
    },
    "outcome_measurements.txt": {
        # Added dispersion_upper_limit_raw based on your diagnostic
        "numeric_cols": ["param_value", "dispersion_value", "dispersion_upper_limit_raw", "dispersion_lower_limit_raw"]
    },
    "outcome_analyses.txt": {
        # Added raw limits based on Mixed Type audit
        "numeric_cols": ["ci_upper_limit_raw", "ci_lower_limit_raw", "p_value"]
    },
    "outcomes.txt": {
        "date_cols": ["anticipated_posting_date"]
    },
    "provided_documents.txt": {
        "date_cols": ["document_date"]
    }
}

In [18]:
def process_file(filename):
    in_path = RAW_DATA_PATH / filename
    out_path = CLEAN_DATA_PATH / filename

    if not in_path.exists():
        print(f"[SKIP] {filename} not found.")
        return

    print(f"Processing: {filename}...", end=" ")

    try:
        # 1. LOAD (Standard Safe Load)
        df = pd.read_csv(in_path, **AACT_LOAD_PARAMS)

        # 2. STANDARD HYGIENE (Apply to ALL files)
        # Strip invisible whitespace from all text columns
        obj_cols = df.select_dtypes(include=['object']).columns
        for col in obj_cols:
            df[col] = df[col].str.strip()

        # 3. SPECIAL CLEANING (Only for specific files)
        if filename in SPECIAL_CONFIG:
            rules = SPECIAL_CONFIG[filename]

            # Apply Numeric Rules
            for col in rules.get("numeric_cols", []):
                if col in df.columns:
                    df[col] = clean_numeric_column(df[col])

            # Apply Date Rules
            for col in rules.get("date_cols", []):
                if col in df.columns:
                    df[col] = clean_date_column(df[col])

            print(f"[Special Cleaning Applied] -> ", end="")
        else:
            print(f"[Standard Cleaning] -> ", end="")

        # 4. EXPORT (Clean Pipe-Delimited TXT)
        # quoting=csv.QUOTE_MINIMAL ensures pipes in text are quoted
        df.to_csv(out_path, index=False, sep='|', quoting=csv.QUOTE_MINIMAL)
        print(f"Saved ({len(df)} rows)")

    except Exception as e:
        print(f"\n  [ERROR] Failed: {e}")

In [19]:
# --- MAIN EXECUTION ---

# Combine your lists to ensure we catch everything
target_files = [
    "studies.txt",
    "sponsors.txt",
    "interventions.txt",
    "countries.txt",
    "designs.txt",
    "calculated_values.txt",
    "conditions.txt",
    "browse_conditions.txt",
    "brief_summaries.txt",
    "design_groups.txt",
    "outcomes.txt",
    "eligibilities.txt",
    "design_outcomes.txt",
    "outcome_analyses.txt",
    # Adding the measurement files you analyzed separately
    "outcome_measurements.txt",
    "baseline_measurements.txt"
]

# Remove duplicates just in case
target_files = sorted(list(set(target_files)))

print(f"Starting Batch Cleaning of {len(target_files)} files...\n")

for filename in target_files:
    process_file(filename)

print(f"\n[DONE] Cleaned files are located in: {CLEAN_DATA_PATH}")

Starting Batch Cleaning of 16 files...

Processing: baseline_measurements.txt... [Special Cleaning Applied] -> Saved (2755628 rows)
Processing: brief_summaries.txt... [Standard Cleaning] -> Saved (563480 rows)
Processing: browse_conditions.txt... [Standard Cleaning] -> Saved (4119582 rows)
Processing: calculated_values.txt... [Standard Cleaning] -> Saved (564443 rows)
Processing: conditions.txt... [Standard Cleaning] -> Saved (1001190 rows)
Processing: countries.txt... [Standard Cleaning] -> Saved (770267 rows)
Processing: design_groups.txt... [Standard Cleaning] -> Saved (1033742 rows)
Processing: design_outcomes.txt... [Standard Cleaning] -> Saved (3483272 rows)
Processing: designs.txt... [Standard Cleaning] -> Saved (559710 rows)
Processing: eligibilities.txt... [Standard Cleaning] -> Saved (563480 rows)
Processing: interventions.txt... [Standard Cleaning] -> Saved (954821 rows)
Processing: outcome_analyses.txt... [Special Cleaning Applied] -> Saved (310455 rows)
Processing: outcome

In [2]:
#!/usr/bin/env python3
import pandas as pd
import csv
import os
import sys
from pathlib import Path

# --- 1. CONFIGURATION ---
# We assume this script is run from the Project Root
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_PATH = DATA_DIR / "raw"
CLEAN_DATA_PATH = DATA_DIR / "cleaned"

# The critical files to audit
TARGET_FILES = [
    # --- High Risk (Complex Data / Known Issues) ---
    "studies.txt",
    "interventions.txt",
    "outcome_measurements.txt",
    "outcome_analyses.txt",
    "design_groups.txt",
    "baseline_measurements.txt",

    # --- Medium Risk (Dates / Long Text) ---
    "outcomes.txt",
    "brief_summaries.txt",
    "eligibilities.txt",
    "conditions.txt",
    "browse_conditions.txt",

    # --- Low Risk (Standard IDs / Categories) ---
    "sponsors.txt",
    "countries.txt",
    "designs.txt",
    "calculated_values.txt",
    "design_outcomes.txt"
]
# --- 2. THE SIMULATOR (Your Exact Loader Logic) ---
class DataLoaderSimulator:
    def __init__(self, data_path):
        self.data_path = data_path

        # --- STRATEGY A: PERFECT (The Goal) ---
        self.params_perfect = {
            "sep": "|", "dtype": str, "header": 0, "quotechar": '"',
            "quoting": csv.QUOTE_MINIMAL, "low_memory": False, "on_bad_lines": "warn"
        }

        # --- STRATEGY B: ROBUST (The Fail-Safe) ---
        self.params_robust = {
            "sep": "|", "dtype": str, "header": 0, "quotechar": '"',
            "quoting": 3, "low_memory": False, "on_bad_lines": "warn"
        }

    def simulate_load(self, filename):
        """
        Tries to load using your logic.
        Returns: (DataFrame, Strategy_Used_Name, Error_Message)
        """
        full_path = self.data_path / filename
        if not full_path.exists():
            return None, "MISSING", "File not found"

        # 1. Try Strategy A
        try:
            df = pd.read_csv(full_path, **self.params_perfect)
            return df, "Strategy A (Perfect)", None
        except Exception as e_perfect:
            # 2. Fallback to Strategy B
            try:
                df = pd.read_csv(full_path, **self.params_robust)
                return df, "Strategy B (Robust)", str(e_perfect)
            except Exception as e_critical:
                return None, "FAILED", str(e_critical)

# --- 3. AUDIT FUNCTIONS ---

def get_raw_line_count(filepath):
    """Fast line counter for raw files (binary mode for speed)."""
    try:
        with open(filepath, 'rb') as f:
            return sum(1 for _ in f) - 1 # Subtract header
    except:
        return 0

def audit_file(filename, simulator):
    raw_path = RAW_DATA_PATH / filename

    # 1. ROW COUNT CHECK
    raw_count = get_raw_line_count(raw_path)

    # 2. LOADER SIMULATION
    df, strategy, error = simulator.simulate_load(filename)

    if df is None:
        print(f"{filename:<28} | {'CRITICAL FAIL':<20} | Raw: {raw_count:<8} | {error}")
        return

    clean_count = len(df)
    diff = raw_count - clean_count

    # Status Logic for Rows
    row_status = f"{clean_count} (Diff: {diff})"
    if diff > 1000:
        row_status += " [LOSS!]"
    elif diff < 0:
        row_status += " [GAIN!]" # Bad splitting usually causes row gains

    # Status Logic for Strategy
    strat_status = strategy
    if "Robust" in strategy:
        strat_status = "WARN: Used B"

    # 3. TYPE CASTING CHECK (The "Lazy Loading" Trap)
    # We verify that strings can actually become numbers/dates
    type_status = "OK"

    # Case A: Numeric Check (outcome_measurements / analyses / baseline)
    if filename in ["outcome_measurements.txt", "outcome_analyses.txt", "baseline_measurements.txt"]:
        # Identify the numeric column
        target_col = None
        if "param_value" in df.columns: target_col = "param_value"
        elif "ci_upper_limit_raw" in df.columns: target_col = "ci_upper_limit_raw"

        if target_col:
            # Attempt conversion
            numerics = pd.to_numeric(df[target_col], errors='coerce')

            # Check if we have strings that aren't empty but failed conversion
            # We treat empty strings/whitespace as valid NaNs. We only care about "Text Garbage".
            non_empty_strings = df[target_col].replace(r'^\s*$', float('nan'), regex=True).dropna()
            valid_numerics = numerics.dropna()

            # If valid count is significantly lower than non-empty count (allow 1% margin)
            if len(valid_numerics) < len(non_empty_strings) * 0.99:
                type_status = f"FAIL (Num Cast {target_col})"

    # Case B: Date Check (studies)
    if filename == "studies.txt":
        if "start_date" in df.columns:
            dates = pd.to_datetime(df["start_date"], errors='coerce')
            if dates.notna().sum() == 0:
                type_status = "FAIL (Date Parse)"
            elif dates.max().year > 2100:
                type_status = "FAIL (Future Date)"

    # PRINT RESULT
    print(f"{filename:<28} | {strat_status:<20} | {row_status:<20} | {type_status}")

# --- 4. MAIN EXECUTION ---

def run_full_audit():
    # Path Verification
    if not CLEAN_DATA_PATH.exists():
        print(f"[ERROR] Cleaned data directory not found at: {CLEAN_DATA_PATH}")
        print("Please ensure you are running this script from the project root.")
        sys.exit(1)

    print("="*100)
    print("FINAL DATA QUALITY ASSURANCE REPORT")
    print("="*100)
    print(f"Project Root: {PROJECT_ROOT}")
    print(f"Checking:     {CLEAN_DATA_PATH}")
    print("-" * 100)
    print(f"{'FILENAME':<28} | {'LOAD STRATEGY':<20} | {'ROWS (Diff)':<20} | {'TYPE SAFETY'}")
    print("-" * 100)

    simulator = DataLoaderSimulator(CLEAN_DATA_PATH)

    for filename in TARGET_FILES:
        audit_file(filename, simulator)

    print("-" * 100)
    print("INTERPRETATION GUIDE:")
    print("1. LOAD STRATEGY: Must be 'Strategy A'. If 'Strategy B', your cleaning failed to fix quotes.")
    print("2. ROWS: Diff should be 0 or small positive (dropped bad rows). Large positive = Data Loss.")
    print("3. TYPE SAFETY: 'OK' means your strings will successfully convert to numbers/dates in ML.")
    print("="*100)

if __name__ == "__main__":
    run_full_audit()

FINAL DATA QUALITY ASSURANCE REPORT
Project Root: /home/delaunan/code/delaunan/clintrialpredict
Checking:     /home/delaunan/code/delaunan/clintrialpredict/data/cleaned
----------------------------------------------------------------------------------------------------
FILENAME                     | LOAD STRATEGY        | ROWS (Diff)          | TYPE SAFETY
----------------------------------------------------------------------------------------------------
studies.txt                  | Strategy A (Perfect) | 564443 (Diff: 0)     | OK
interventions.txt            | Strategy A (Perfect) | 954821 (Diff: 0)     | OK
outcome_measurements.txt     | Strategy A (Perfect) | 4679165 (Diff: 0)    | OK
outcome_analyses.txt         | Strategy A (Perfect) | 310455 (Diff: 0)     | OK
design_groups.txt            | Strategy A (Perfect) | 1033742 (Diff: 0)    | OK
baseline_measurements.txt    | Strategy A (Perfect) | 2755628 (Diff: 0)    | OK
outcomes.txt                 | Strategy A (Perfect) | 627033

In [1]:
import pandas as pd
import numpy as np
import sys
import os
from pathlib import Path
from scipy.stats import pointbiserialr

# --- 1. SETUP PATHS & IMPORT ---
PROJECT_ROOT = Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "cleaned"
OUTPUT_FILE = PROJECT_ROOT / "Field_Extraction_Audit.txt"

# Add src/prep to the system path so we can import the loader
loader_path = PROJECT_ROOT / "src" / "prep"
if str(loader_path) not in sys.path:
    sys.path.append(str(loader_path))

try:
    from data_loader import ClinicalTrialLoader
    print(f"[SUCCESS] Imported ClinicalTrialLoader from {loader_path}")
except ImportError as e:
    print(f"[CRITICAL] Could not import 'ClinicalTrialLoader'. Error: {e}")
    print(f"Ensure 'data_loader.py' exists in: {loader_path}")
    sys.exit(1)

# --- 2. AUDIT UTILITIES ---
class AuditLogger:
    def __init__(self, filepath):
        self.filepath = filepath
        self.buffer = []

    def log(self, msg):
        print(msg)
        self.buffer.append(msg)

    def section(self, title):
        line = "=" * 80
        self.log(f"\n{line}\n{title.upper()}\n{line}")

    def save(self):
        with open(self.filepath, "w", encoding="utf-8") as f:
            f.write("\n".join(self.buffer))
        print(f"\n[DONE] Full audit report saved to: {self.filepath}")

logger = AuditLogger(OUTPUT_FILE)

def analyze_numeric(df, col, target_col='target'):
    """Detailed stats for numeric features."""
    if col not in df.columns:
        logger.log(f"   [MISSING] Column '{col}' not found.")
        return

    series = df[col]
    n_total = len(series)
    n_miss = series.isna().sum()
    n_zeros = (series == 0).sum()

    # Stats
    mean_val = series.mean()
    median_val = series.median()
    std_val = series.std()
    min_val = series.min()
    max_val = series.max()

    # Correlation with Target
    corr = "N/A"
    if series.nunique() > 1 and target_col in df.columns:
        try:
            # Drop NaNs for correlation calculation
            valid = df[[col, target_col]].dropna()
            if not valid.empty:
                res = pointbiserialr(valid[col], valid[target_col])
                corr = f"{res.statistic:.4f}"
        except:
            corr = "Err"

    logger.log(f"   FIELD: {col:<28} | Type: Numeric")
    logger.log(f"     - Fill Rate:   {100*(1-n_miss/n_total):.1f}% (Missing: {n_miss})")
    logger.log(f"     - Zero Rate:   {100*(n_zeros/n_total):.1f}% (Sparsity)")
    logger.log(f"     - Stats:       Mean={mean_val:.2f} | Med={median_val:.2f} | Max={max_val:.2f} | Std={std_val:.2f}")
    logger.log(f"     - Target Corr: {corr} (Watch for > 0.8)")

    # Logic Checks
    if std_val == 0:
        logger.log("     [!] WARNING: Constant value (Variance = 0). Drop this feature.")
    if corr != "N/A" and corr != "Err" and abs(float(corr)) > 0.9:
        logger.log("     [!] CRITICAL: Potential Data Leakage (Corr > 0.9).")

def analyze_categorical(df, col):
    """Detailed stats for categorical features."""
    if col not in df.columns:
        logger.log(f"   [MISSING] Column '{col}' not found.")
        return

    series = df[col].fillna("MISSING")
    counts = series.value_counts()
    top_3 = counts.head(3).to_dict()

    logger.log(f"   FIELD: {col:<28} | Type: Categorical")
    logger.log(f"     - Unique Vals: {series.nunique()}")
    logger.log(f"     - Top Values:  {top_3}")

    if "MISSING" in counts:
        pct_miss = (counts["MISSING"] / len(df)) * 100
        logger.log(f"     - Missing %:   {pct_miss:.1f}%")
        if pct_miss > 50:
            logger.log("     [!] WARNING: High missingness.")

    if series.nunique() == 1:
        logger.log("     [!] WARNING: Constant value. Drop this feature.")

# --- 3. MAIN EXECUTION ---
def run_audit():
    logger.section("1. INITIALIZATION & LOADING")
    logger.log(f"Data Path: {DATA_PATH}")

    if not DATA_PATH.exists():
        logger.log("[FATAL] Data path does not exist.")
        return

    # Initialize Loader
    loader = ClinicalTrialLoader(DATA_PATH)

    # --- PHASE 1: LOAD & CLEAN ---
    try:
        df = loader.load_and_clean()
        logger.log(f"Core Cohort Loaded: {len(df)} rows")
    except Exception as e:
        logger.log(f"[FATAL] Loader crashed during load_and_clean: {e}")
        return

    # --- PHASE 2: FEATURE ENGINEERING ---
    try:
        df = loader.add_features(df)
        logger.log(f"Feature Engineering Complete. Total Columns: {len(df.columns)}")
    except Exception as e:
        logger.log(f"[FATAL] Loader crashed during add_features: {e}")
        # We continue with whatever df we have to debug

    # --- PHASE 3: FIELD-BY-FIELD AUDIT ---

    # A. TARGET & CORE
    logger.section("2. CORE IDENTIFIERS & TARGET")
    analyze_categorical(df, 'overall_status')
    analyze_numeric(df, 'target')
    analyze_numeric(df, 'start_year')
    analyze_categorical(df, 'phase_group')
    analyze_numeric(df, 'includes_us')

    # B. CLINICAL SETTING (Regex Check)
    logger.section("3. CLINICAL SETTING (Regex Health)")
    # Check if regex is actually catching things
    for col in ['is_acute', 'is_refractory', 'is_severe']:
        analyze_numeric(df, col)

    # Logic Check: Are we catching anything at all?
    total_clinical_signal = df[['is_acute', 'is_refractory', 'is_severe']].sum().sum()
    if total_clinical_signal == 0:
        logger.log("   [!] CRITICAL: All Clinical Setting flags are 0. Regex patterns failed.")

    # C. COMPARATOR (Design Groups)
    logger.section("4. COMPARATOR ARCHITECTURE")
    analyze_numeric(df, 'has_placebo')
    analyze_numeric(df, 'has_active_comparator')

    # D. DURATION (Waterfall Logic)
    logger.section("5. PROTOCOL DURATION")
    analyze_numeric(df, 'duration_months')
    # Specific check for Imputation
    median_val = df['duration_months'].median()
    imputed_count = (df['duration_months'] == median_val).sum()
    pct_imputed = imputed_count/len(df)
    logger.log(f"   [Logic Check] {imputed_count} rows ({pct_imputed:.1%}) match the median exactly.")
    if pct_imputed > 0.6:
        logger.log("                 [!] WARNING: >60% Median Imputation. Waterfall extraction is weak.")

    # E. SMART PATTERNS (Scores)
    logger.section("6. SMART PATTERNS (Rigor & Strictness)")
    analyze_numeric(df, 'design_rigor_score')
    analyze_numeric(df, 'eligibility_strictness_score')
    analyze_numeric(df, 'is_sick_only')
    analyze_numeric(df, 'criteria_len_log')
    analyze_numeric(df, 'num_primary_endpoints')

    # F. AGENT TYPE (The Classifier)
    logger.section("7. AGENT CLASSIFICATION")
    analyze_categorical(df, 'agent_category')
    # Check fallback rate
    fallback_rate = (df['agent_category'] == 'SMALL_MOLECULE_OTHER').mean()
    logger.log(f"   [Logic Check] 'SMALL_MOLECULE_OTHER' Rate: {fallback_rate:.1%}")
    if fallback_rate > 0.75:
        logger.log("                 [!] WARNING: >75% Fallback. Regex patterns for drugs might be too narrow.")

    # G. COMPETITION (Rolling Density)
    logger.section("8. COMPETITION METRICS")
    analyze_numeric(df, 'competition_broad')
    analyze_numeric(df, 'competition_niche')
    analyze_numeric(df, 'competition_agent')

    # Check for calculation failure
    if df['competition_broad'].sum() == 0:
        logger.log("   [!] CRITICAL: Competition metrics are all 0. Check 'start_year' or 'therapeutic_area'.")

    # H. SPONSOR
    logger.section("9. SPONSOR TIERS")
    analyze_categorical(df, 'sponsor_tier')
    analyze_categorical(df, 'agency_class')
    analyze_categorical(df, 'lead_sponsor')

    # I. MEDICAL HIERARCHY
    logger.section("10. MEDICAL HIERARCHY")
    analyze_categorical(df, 'therapeutic_area')
    analyze_categorical(df, 'therapeutic_subgroup_name')
    # Check Unclassified rate
    unclass_rate = (df['therapeutic_area'] == 'Unclassified').mean()
    logger.log(f"   [Logic Check] Unclassified Rate: {unclass_rate:.1%}")

    # J. TEXT FEATURES
    logger.section("11. NLP TEXT PILLARS")
    for col in ['txt_scientific_essence', 'txt_criteria', 'txt_primary_endpoints']:
        if col in df.columns:
            avg_len = df[col].astype(str).str.len().mean()
            logger.log(f"   FIELD: {col:<28} | Avg Length: {avg_len:.0f} chars")
            if avg_len < 50:
                logger.log("     [!] WARNING: Text is suspiciously short.")
        else:
            logger.log(f"   [MISSING] {col}")

    # K. SCIENTIFIC SUCCESS (P-Values)
    logger.section("12. SCIENTIFIC SUCCESS (P-Values)")
    analyze_numeric(df, 'min_p_value')
    analyze_numeric(df, 'scientific_success')

    # L. SAFE FEATURES
    logger.section("13. REGULATORY GATES")
    analyze_numeric(df, 'has_dmc')
    analyze_numeric(df, 'is_fda_regulated_drug')

    # M. AGE FLAGS
    logger.section("14. AGE DEMOGRAPHICS")
    analyze_numeric(df, 'child')
    analyze_numeric(df, 'adult')
    analyze_numeric(df, 'older_adult')

    # Save Report
    logger.save()

if __name__ == "__main__":
    run_audit()

[SUCCESS] Imported ClinicalTrialLoader from /home/delaunan/code/delaunan/clintrialpredict/src/prep

1. INITIALIZATION & LOADING
Data Path: /home/delaunan/code/delaunan/clintrialpredict/data/cleaned
>>> 1. Loading Studies & Applying Filters...
    [Filter] Kept 126474 Industry-led trials.
    [Sanitizer] Dropping 169 trials terminated due to COVID/Logistics.
    Core Cohort: 34155 trials (Phase 1/2/3, 2005-2025 training window and 2005-2025 for production).
Core Cohort Loaded: 34155 rows
>>> 2. Engineering Features...
    -> Grouping Phases into Efficacy Tiers...
    -> Engineering Clinical Setting (Deep Search)...
    -> Engineering Comparator Architecture (Robust)...
    -> Engineering Protocol Duration (Waterfall)...
    -> Engineering Sponsor Tiers...
    -> Engineering Protocol Complexity (Calculating Age Flags)...
    -> Attaching Medical Hierarchy (Preserving Tree Structure)...
    -> Engineering NLP Text Pillars (Scientific & Operational)...
    -> Engineering Agent Type (Priori